In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

catalogo = "databricks_cata_managed"

tabela_silver = f"{catalogo}.silver.cotacao_moeda"
tabela_gold_fato = f"{catalogo}.gold.fato_cotacao_diaria"
tabela_gold_indicador = f"{catalogo}.gold.fato_indicador_moeda"

df_silver = spark.table(tabela_silver)
df_gold = spark.table(tabela_gold_fato)


In [0]:

df_gold = spark.table(tabela_gold_fato)

df_indicador = (
    df_gold
    .groupBy("codigo_moeda")
    .agg(
        F.count("*").alias("dias_cotados"),
        F.avg("media_venda").cast("decimal(18,6)").alias("cotacao_media_periodo"),
        F.max("maior_venda").alias("maior_cotacao"),
        F.min("menor_venda").alias("menor_cotacao")
    )
    .withColumn("data_cotacao", F.current_date())
    .withColumn("_data_atualizacao", F.current_timestamp())
    .select(
        "codigo_moeda",
        "data_cotacao",
        "dias_cotados",
        "cotacao_media_periodo",
        "maior_cotacao",
        "menor_cotacao",
        "_data_atualizacao"
    )
)


In [0]:
if not spark.catalog.tableExists(tabela_gold_indicador):
    (
        df_indicador.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(tabela_gold_indicador)
    )
else:
    delta_indicador = DeltaTable.forName(spark, tabela_gold_indicador)

    (
        delta_indicador.alias("destino")
        .merge(
            df_indicador.alias("origem"),
            """
            destino.codigo_moeda = origem.codigo_moeda
            AND destino.data_cotacao = origem.data_cotacao
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )